In [51]:
NO_RECORDS = 102 # more than 100 to test chunking

In [52]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
import os
from fastapi import FastAPI, Query, HTTPException
import logging

from lxml import etree as ET
# API calls
import requests
from lxml import etree


from utils.enrichment_methods import get_doi, clean_doi

In [53]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [54]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")

if (host == '') | (host == "127.0.0.1"):
    host = "127.0.0.1"
    user = os.getenv("POSTGRES_ADMIN")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [55]:
query_records = """
            -- Exploring subjects from record raw metadata

WITH query AS (
    SELECT
        records.*,
        ARRAY(
            SELECT elem->>'subject'
            FROM jsonb_array_elements(datacite_json->'subjects') AS elem
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    doi,
    subjects
FROM query
WHERE 
array_length(subjects, 1) > 0
ORDER BY id
LIMIT %s
;
"""

In [56]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results[0])

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_dois = [row["doi"] for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]
        record_subjects = [set(row["subjects"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch records")
    raise HTTPException(status_code=500, detail=str(e))

{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/10', 'title': 'DEKISO C17H18Cl4', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icat.isis.stfc.ac.uk:inv/10</identifier>\n    <datestamp>2007-09-12T12:59:58Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.3</schemaVersion>\n      <datacentreSymbol>BL.ISIS</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.3/metadata.xsd">\n          <identifier>https://data.isis.stfc.ac.uk/browse/instrument/16/facilityCycle/115101366/investigation/10</identifier>\n          <alternateIdentifiers alternateIdentifi

In [57]:
record_subjects

[{'1989',
  'Bristol',
  'C17H18Cl4',
  'DEKISO',
  'JAKH',
  'JAKHoward',
  'RAL',
  'SXD',
  'ccw',
  'w=90'},
 {'1623',
  '1990',
  'CROSTAT',
  'Dubna',
  'Empty',
  'JINR',
  'SXD',
  'Savenko',
  'centre',
  'stick'},
 {'(La', '1997', 'ISIS', 'Keen', 'Pb)MnO3', 'SXD', 'w=-159'},
 {'Chemistry', 'Materials', 'Physics'},
 {'+0.5A',
  '-0.172A',
  '0',
  '0kV',
  '2.5kV',
  '5kV',
  'Cooling',
  'DC',
  'E-field',
  'Engineering',
  'F=0',
  'F=10',
  'F=100',
  'F=1000',
  'F=10000',
  'F=10100',
  'F=10200',
  'F=10300',
  'F=10400',
  'F=10500',
  'F=10600',
  'F=10700',
  'F=10800',
  'F=10900',
  'F=11000',
  'F=11000.0',
  'F=11100',
  'F=11200',
  'F=11300',
  'F=11400',
  'F=11500',
  'F=11600',
  'F=11700',
  'F=11800',
  'F=11900',
  'F=1200',
  'F=12000',
  'F=12100',
  'F=12200',
  'F=12300',
  'F=12400',
  'F=12500',
  'F=12600',
  'F=12700',
  'F=12800',
  'F=12900',
  'F=13000',
  'F=13100',
  'F=13200',
  'F=13300',
  'F=13400',
  'F=13500',
  'F=13600',
  'F=13700',


# Criteria for subject exclusion: 
* all numbers
* character length < 3
    * example "F=10300" -> number of LETTERS must be >=3?
* example "VSM2=0.729999999999999" -> must not have = in subject
* character length > 25 ?
* remove/reformat &lt &gt &amp

In [58]:
def remove_poor_subjects(subject_set: set[str]):
    retained_subjects = []

    # For each subject, go through the rules
    # If the subject passes all rules, retain it.
    for subject in subject_set:
        is_only_numbers = subject.isdigit()
        is_too_short = len(subject) < 3
        is_too_long = len(subject) > 25

        has_few_letters = sum(i.isalpha() for i in subject) < 3
        contains_equals = "=" in subject

        # no condition should be true
        if not (is_only_numbers or is_too_short or is_too_long or has_few_letters or contains_equals):
            retained_subjects.append(subject)


    retained_subjects_set = set(retained_subjects)
    return retained_subjects_set

In [59]:
cleaned_sets = []
for subs in record_subjects:
    cleaned_sets.append(remove_poor_subjects(subs))

In [60]:
for ix, subs in enumerate(record_subjects):
    print(str(ix) + ". record with " + str(len(subs)) + " subjects now has " + str(len(cleaned_sets[ix])) + " subjects.")

0. record with 10 subjects now has 8 subjects.
1. record with 10 subjects now has 8 subjects.
2. record with 7 subjects now has 4 subjects.
3. record with 3 subjects now has 3 subjects.
4. record with 277 subjects now has 16 subjects.
5. record with 28 subjects now has 15 subjects.
6. record with 16 subjects now has 12 subjects.
7. record with 13 subjects now has 6 subjects.
8. record with 9 subjects now has 6 subjects.
9. record with 179 subjects now has 7 subjects.
10. record with 70 subjects now has 13 subjects.
11. record with 373 subjects now has 9 subjects.
12. record with 9 subjects now has 3 subjects.
13. record with 4 subjects now has 3 subjects.
14. record with 8 subjects now has 6 subjects.
15. record with 6 subjects now has 5 subjects.
16. record with 6 subjects now has 5 subjects.
17. record with 16 subjects now has 6 subjects.
18. record with 165 subjects now has 164 subjects.
19. record with 84 subjects now has 37 subjects.
20. record with 88 subjects now has 6 subjects.

In [61]:
record_subjects[4]

{'+0.5A',
 '-0.172A',
 '0',
 '0kV',
 '2.5kV',
 '5kV',
 'Cooling',
 'DC',
 'E-field',
 'Engineering',
 'F=0',
 'F=10',
 'F=100',
 'F=1000',
 'F=10000',
 'F=10100',
 'F=10200',
 'F=10300',
 'F=10400',
 'F=10500',
 'F=10600',
 'F=10700',
 'F=10800',
 'F=10900',
 'F=11000',
 'F=11000.0',
 'F=11100',
 'F=11200',
 'F=11300',
 'F=11400',
 'F=11500',
 'F=11600',
 'F=11700',
 'F=11800',
 'F=11900',
 'F=1200',
 'F=12000',
 'F=12100',
 'F=12200',
 'F=12300',
 'F=12400',
 'F=12500',
 'F=12600',
 'F=12700',
 'F=12800',
 'F=12900',
 'F=13000',
 'F=13100',
 'F=13200',
 'F=13300',
 'F=13400',
 'F=13500',
 'F=13600',
 'F=13700',
 'F=13800',
 'F=13900',
 'F=1400',
 'F=14000',
 'F=14000.0',
 'F=14100',
 'F=14200',
 'F=14300',
 'F=14400',
 'F=14500',
 'F=14600',
 'F=14700',
 'F=14800',
 'F=14900',
 'F=150',
 'F=1500',
 'F=15000',
 'F=15100',
 'F=15200',
 'F=15300',
 'F=15400',
 'F=15500',
 'F=15600',
 'F=15700',
 'F=15800',
 'F=15900',
 'F=1600',
 'F=16000',
 'F=16100',
 'F=16200',
 'F=16300',
 'F=16400',

In [62]:
cleaned_sets[4]

{'Cooling',
 'E-field',
 'Engineering',
 'HSM:',
 'Quartz',
 'Uncatalyzed',
 'VSM:',
 'air',
 'composite',
 'epoxy',
 'extra',
 'insert',
 'resin',
 'run',
 'second',
 'window'}

In [64]:
record_subjects[11]

{'-20N',
 '10',
 '4',
 '4x4x10',
 'Bing',
 'Cambridge',
 'CeO2',
 'Energy',
 'Engineering',
 'Graphite',
 'Materials',
 'PP',
 'SNG722_1_RT_VT;s-1002.06;e-0.00;p-21.35',
 'SNG722_1_RT_VT;s-1002.37;e-0.00;p-21.37',
 'SNG722_1_RT_VT;s-1003.10;e-0.00;p-21.15',
 'SNG722_1_RT_VT;s-1003.15;e-0.00;p-20.99',
 'SNG722_1_RT_VT;s-1005.13;e-0.00;p-21.39',
 'SNG722_1_RT_VT;s-1005.77;e-0.00;p-21.28',
 'SNG722_1_RT_VT;s-1492.02;e-0.00;p-21.74',
 'SNG722_1_RT_VT;s-1496.26;e-0.00;p-21.41',
 'SNG722_1_RT_VT;s-1496.27;e-0.00;p-21.05',
 'SNG722_1_RT_VT;s-1496.83;e-0.00;p-21.44',
 'SNG722_1_RT_VT;s-1497.37;e-0.00;p-21.20',
 'SNG722_1_RT_VT;s-1497.92;e-0.00;p-21.41',
 'SNG722_1_RT_VT;s-1498.25;e-0.00;p-21.25',
 'SNG722_1_RT_VT;s-1498.83;e-0.00;p-21.52',
 'SNG722_1_RT_VT;s-1501.40;e-0.00;p-21.18',
 'SNG722_1_RT_VT;s-1501.86;e-0.00;p-20.49',
 'SNG722_1_RT_VT;s-1503.84;e-0.00;p-21.38',
 'SNG722_1_RT_VT;s-1993.41;e-0.00;p-21.55',
 'SNG722_1_RT_VT;s-1995.03;e-0.00;p-21.44',
 'SNG722_1_RT_VT;s-1995.46;e-0.00;p-21

In [63]:
cleaned_sets[11]

{'Bing',
 'Cambridge',
 'CeO2',
 'Energy',
 'Engineering',
 'Graphite',
 'Materials',
 'Test',
 'settings'}

## Other adjustments

- some records have more than 200 subjects ---- explore, potentially fixable in transformation?
- lowercase everything (we have ISIS and isis as separate) -- if the set of keywords and lowercase keywords have different lengths, they neeed triming 